# CAP182 SS2 Common Issues & Key Concepts

This notebook addresses four recurring issues from the SS2 assessment.  
We use the **Bank Marketing** dataset from the UCI Machine Learning Repository throughout.

**Contents**
1. Data Leakage — the `duration` trap
2. What `class_weight` actually does
3. Confidence Intervals for a single model
4. Statistical comparison between models

In [0]:
# Install dependencies
!pip install ucimlrepo xgboost -q 

In [0]:
import numpy as np
import pandas as pd
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score,
    RepeatedStratifiedKFold
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report
)
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
print('All imports successful ✓')

## 1. Load the Bank Marketing dataset

In [0]:
from ucimlrepo import fetch_ucirepo

dataset = fetch_ucirepo(id=222)
df = dataset.data.original.copy()

print(f'Shape: {df.shape}')
df.head()

## 2. Exploratory Data Analysis (EDA) on the Bank Marketing dataset

In [0]:
# 1. Basic structure
print('--- Shape ---')
print(df.shape)
print()
print('--- Column names & dtypes ---')
print(df.dtypes)
print()
print('--- First 5 rows ---')
display(df.head())

# 2. Missing values (NaN) per column
print('--- Missing values (NaN) per column ---')
print(df.isna().sum())
print()

# 3. Summary statistics — numeric features
print('--- Numeric summary statistics ---')
display(df.describe())

# 4. Summary statistics — categorical features
print('--- Categorical summary statistics ---')
display(df.describe(include=['object', 'category']))

# 5. Target distribution
y_col = 'y'  # subscription to a term deposit (yes / no)
print(f'--- Target ({y_col}) distribution ---')
print(df[y_col].value_counts())
print()
print(df[y_col].value_counts(normalize=True).round(4) * 100, '%')
print()

# 6. Categorical feature value counts (quick scan)
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
for col in cat_cols:
    print(f'--- {col} ({df[col].nunique()} unique) ---')
    print(df[col].value_counts())
    print()

# 7. Numeric feature distributions by target
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print('--- Numeric features grouped by target ---')
display(df.groupby(y_col)[num_cols].agg(['mean', 'median', 'std']))

# 8. Correlation matrix among numeric features (with legend)
print('--- Correlation matrix (numeric features) ---')
num_df = df.select_dtypes(include=[np.number])
corr = num_df.corr()

styled_corr = (
    corr.style
    .background_gradient(cmap='coolwarm', axis=None, vmin=-1, vmax=1)
    .format(precision=2)
    .set_caption(
        'Correlation matrix — colour scale: 🔴 +1.00 (strong positive) ↔ '
        '⚪ 0.00 (no correlation) ↔ 🔵 −1.00 (strong negative)'
    )
)
display(styled_corr)

# 9. Unknown / placeholder values in categorical columns
print('--- "unknown" value counts in categorical columns ---')
for col in cat_cols:
    unk = (df[col] == 'unknown').sum()
    if unk > 0:
        print(f'{col}: {unk} ({unk / len(df) * 100:.1f}%)')


In [0]:
# 10. Duration vs target — the data-leakage smoking gun
# `duration` is still in df at this point (it gets dropped in preprocessing).
# A long call almost always means the client said "yes", so this column
# leaks the answer into the features.
print('Average call duration by outcome:\n')
print(df.groupby('y')['duration'].agg(['mean', 'median', 'std']).round(1))

## 3. Data Preprocessing

In [0]:
# 3a. Drop the `duration` column — DATA LEAKAGE
# `duration` = length of the last phone call (in seconds).  A call only has a
# duration *after* it ends, and a long call strongly implies the client said
# "yes".  Including it inflates every metric but is impossible to know at
# prediction time, so we remove it.  (This is explored in Section 4 below.)
if 'duration' in df.columns:
    df = df.drop(columns=['duration'])
    print('Dropped `duration` (data leakage) ✓')

# 3b. Separate features (X) and target (y)
y_col = 'y'
X = df.drop(columns=[y_col])
y = df[y_col]                     # values: 'yes' / 'no'  (sklearn accepts strings natively)

print(f'\nFeatures shape : {X.shape}')
print(f'Target shape   : {y.shape}')
print(f'Target classes : {y.unique().tolist()}')
print(f'Class counts   :\n{y.value_counts()}')

# 3c. Identify column types
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f'\nNumeric features     ({len(numeric_features)}): {numeric_features}')
print(f'Categorical features ({len(categorical_features)}): {categorical_features}')

# 3d. Build a preprocessing pipeline with ColumnTransformer
#   - Numeric:    median imputation  →  StandardScaler
#   - Categorical: impute 'unknown'  →  OneHotEncoder (handle 'unknown' as its own category)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ],
    remainder='drop',
)

print('\nPreprocessor built ✓')

# 3e. Stratified train / test split (80 / 20)
# The target is imbalanced (~11 % 'yes'), so stratification preserves the ratio.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f'\nTrain set: X_train {X_train.shape}, y_train {y_train.shape}')
print(f'Test  set: X_test  {X_test.shape},  y_test  {y_test.shape}')
print(f'\nTrain class proportions:\n{y_train.value_counts(normalize=True).round(4)}')
print(f'\nTest  class proportions:\n{y_test.value_counts(normalize=True).round(4)}')

# 3f. Fit the preprocessor on the TRAINING data only (avoids leakage)
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Retrieve the expanded feature names from the fitted OneHotEncoder
ohe = preprocessor.named_transformers_['cat'].named_steps['onehot']
cat_feature_names = ohe.get_feature_names_out(categorical_features).tolist()
processed_feature_names = numeric_features + cat_feature_names

print(f'\nProcessed feature count: {len(processed_feature_names)}')
print(f'Processed train shape : {X_train_processed.shape}')
print(f'Processed test shape  : {X_test_processed.shape}')
print('\nPreprocessing complete ✓')

## 3. Feature Engineering

In [0]:
# ---------------------------------------------------------------------------
# Feature Engineering
# We create new features BEFORE the ColumnTransformer so that the preprocessor
# can encode / scale them along with the originals.  All transformations are
# applied to BOTH train and test, and nothing here looks at y, so there is no
# leakage.
# ---------------------------------------------------------------------------

# Work on copies so the originals stay intact for comparison
X_train_fe = X_train.copy()
X_test_fe = X_test.copy()

# --- 1. Age bins ----------------------------------------------------------
# Converts a continuous variable into meaningful life-stage buckets.
age_bins = [0, 25, 35, 45, 55, 65, 120]
age_labels = ['<25', '25-34', '35-44', '45-54', '55-64', '65+']
X_train_fe['age_group'] = pd.cut(X_train_fe['age'], bins=age_bins, labels=age_labels)
X_test_fe['age_group'] = pd.cut(X_test_fe['age'], bins=age_bins, labels=age_labels)

# --- 2. Balance categories ------------------------------------------------
# Split account balance into intuitive bands; captures non-linear effects.
balance_bins = [-float('inf'), 0, 500, 2000, 10000, float('inf')]
balance_labels = ['negative', '0-500', '500-2k', '2k-10k', '10k+']
X_train_fe['balance_group'] = pd.cut(X_train_fe['balance'], bins=balance_bins, labels=balance_labels)
X_test_fe['balance_group'] = pd.cut(X_test_fe['balance'], bins=balance_bins, labels=balance_labels)

# --- 3. Has been previously contacted -------------------------------------
# pdays == -1 means the client was never contacted before.  Binary flag.
X_train_fe['was_contacted_before'] = (X_train_fe['pdays'] != -1).astype(int)
X_test_fe['was_contacted_before'] = (X_test_fe['pdays'] != -1).astype(int)

# --- 4. Total contacts across campaigns -----------------------------------
# campaign = contacts during this campaign; previous = contacts before this
# campaign.  Their sum is the overall effort spent on the client.
X_train_fe['total_contacts'] = X_train_fe['campaign'] + X_train_fe['previous']
X_test_fe['total_contacts'] = X_test_fe['campaign'] + X_test_fe['previous']

# --- 5. Contact success rate ----------------------------------------------
# previous >= 1 implies at least one prior contact; if previous is 0 the rate
# is undefined, so we guard with a denominator (set to 1 and flag).
X_train_fe['prior_success_rate'] = np.where(
    X_train_fe['previous'] > 0,
    0.0,  # placeholder; poutcome provides outcome info, rate computed below
    0.0,
)
# More useful: flag whether the previous outcome was a success
X_train_fe['prior_success'] = (X_train_fe['poutcome'] == 'success').astype(int)
X_test_fe['prior_success'] = (X_test_fe['poutcome'] == 'success').astype(int)

# Drop the placeholder column created above (kept for illustration)
if 'prior_success_rate' in X_train_fe.columns:
    X_train_fe = X_train_fe.drop(columns=['prior_success_rate'])
if 'prior_success_rate' in X_test_fe.columns:
    X_test_fe = X_test_fe.drop(columns=['prior_success_rate'])

# --- 6. Season from month --------------------------------------------------
# Map month string to a season to reduce cardinality.
month_to_season = {
    'jan': 'winter', 'feb': 'winter', 'dec': 'winter',
    'mar': 'spring', 'apr': 'spring', 'may': 'spring',
    'jun': 'summer', 'jul': 'summer', 'aug': 'summer',
    'sep': 'autumn', 'oct': 'autumn', 'nov': 'autumn',
}
X_train_fe['season'] = X_train_fe['month'].str.lower().map(month_to_season).fillna('unknown')
X_test_fe['season'] = X_test_fe['month'].str.lower().map(month_to_season).fillna('unknown')

# --- 7. Financial burden indicator ----------------------------------------
# Client has both a housing loan and a personal loan → higher financial burden.
X_train_fe['has_both_loans'] = (
    (X_train_fe['housing'] == 'yes') & (X_train_fe['loan'] == 'yes')
).astype(int)
X_test_fe['has_both_loans'] = (
    (X_test_fe['housing'] == 'yes') & (X_test_fe['loan'] == 'yes')
).astype(int)

# --- 8. Balance-to-contacts ratio -----------------------------------------
# How much account balance per contact attempt — a rough "effort efficiency"
# proxy.  Guard against division by zero.
X_train_fe['balance_per_contact'] = X_train_fe['balance'] / (X_train_fe['campaign'].replace(0, 1))
X_test_fe['balance_per_contact'] = X_test_fe['balance'] / (X_test_fe['campaign'].replace(0, 1))

# --- Summary ---------------------------------------------------------------
new_features = [
    'age_group', 'balance_group', 'was_contacted_before', 'total_contacts',
    'prior_success', 'season', 'has_both_loans', 'balance_per_contact',
]
print(f'Original feature count : {X_train.shape[1]}')
print(f'Engineered feature count: {X_train_fe.shape[1]}')
print(f'New features added      : {len(new_features)}')
print(f'\nNew feature names:')
for f in new_features:
    print(f'  • {f}')

print('\nSample of engineered features:')
display(X_train_fe[new_features].head(10))

# --- Rebuild preprocessor to include new features --------------------------
# Update the column-type lists so the ColumnTransformer picks up the new cols.
numeric_features_fe = X_train_fe.select_dtypes(include=[np.number]).columns.tolist()
categorical_features_fe = X_train_fe.select_dtypes(include=['object', 'category']).columns.tolist()

print(f'\nNumeric features (FE)     ({len(numeric_features_fe)}): {numeric_features_fe}')
print(f'Categorical features (FE) ({len(categorical_features_fe)}): {categorical_features_fe}')

preprocessor_fe = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features_fe),
        ('cat', categorical_transformer, categorical_features_fe),
    ],
    remainder='drop',
)

# Fit on train, transform both — no leakage
X_train_fe_processed = preprocessor_fe.fit_transform(X_train_fe)
X_test_fe_processed = preprocessor_fe.transform(X_test_fe)

print(f'\nProcessed train shape (FE): {X_train_fe_processed.shape}')
print(f'Processed test  shape (FE): {X_test_fe_processed.shape}')
print('\nFeature engineering complete ✓')

## 3. Modelling

In [0]:
# ===========================================================================
# 3. Modelling — Logistic Regression, Random Forest, XGBoost
# Class weighting for imbalanced targets, cross-validation, confidence
# intervals, and statistical model comparison.
# ===========================================================================
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder

# --- Encode string labels to 0/1 for XGBoost -------------------------------
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)   # 'no'→0, 'yes'→1
y_test_enc  = le.transform(y_test)

class_names = le.classes_.tolist()       # ['no', 'yes']
pos_label = 1                              # 'yes' is the positive (minority) class
print(f'Label encoding: {dict(zip(class_names, range(len(class_names))))}')

# --- Class-weight dictionary (sklearn models) ------------------------------
# The dataset is ~11 % 'yes'.  Class weights inversely proportional to frequency
# force the model to pay more attention to the minority class.
from collections import Counter
train_counts = Counter(y_train_enc)
sample_weights_train = np.array([
    len(y_train_enc) / (len(class_names) * train_counts[c])
    for c in y_train_enc
])

class_weight_dict = {
    c: len(y_train_enc) / (len(class_names) * train_counts[c])
    for c in range(len(class_names))
}
print(f'Class weights: {class_weight_dict}')

# --- Define models ---------------------------------------------------------
models = {
    'Logistic Regression': LogisticRegression(
        class_weight=class_weight_dict,
        max_iter=1000,
        random_state=RANDOM_STATE,
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=300,
        learning_rate=0.1,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        # scale_pos_weight = neg_count / pos_count handles imbalance in XGBoost
        scale_pos_weight=train_counts[0] / train_counts[1],
        eval_metric='logloss',
        random_state=RANDOM_STATE,
        n_jobs=-1,
        use_label_encoder=False,
    ),
}

# ===========================================================================
# 3a. Train each model and evaluate on the held-out test set
# ===========================================================================
results = {}

print('=' * 70)
print('Test-set evaluation')
print('=' * 70)
for name, model in models.items():
    model.fit(X_train_processed, y_train_enc)
    y_pred = model.predict(X_test_processed)
    acc = accuracy_score(y_test_enc, y_pred)
    f1  = f1_score(y_test_enc, y_pred, pos_label=pos_label)
    results[name] = {'model': model, 'y_pred': y_pred, 'accuracy': acc, 'f1': f1}
    print(f'\n{name}')
    print(f'  Accuracy : {acc:.4f}')
    print(f'  F1 (yes) : {f1:.4f}')
    print(classification_report(y_test_enc, y_pred, target_names=class_names))

# ===========================================================================
# 3b. Cross-validation with confidence intervals
# ===========================================================================
# RepeatedStratifiedKFold gives a more stable estimate of model performance.
# We compute the mean CV score and a 95 % confidence interval using the
# t-distribution (appropriate for a small number of CV folds).

rkf = RepeatedStratifiedKFold(
    n_splits=5, n_repeats=10, random_state=RANDOM_STATE
)

print('=' * 70)
print('Cross-validation (5-fold × 10 repeats) with 95 % confidence intervals')
print('=' * 70)

cv_results = {}
for name, model in models.items():
    # Clone the model so each CV run starts fresh
    from sklearn.base import clone
    scores = cross_val_score(
        clone(model), X_train_processed, y_train_enc,
        cv=rkf, scoring='f1', n_jobs=-1,
    )
    mean_score = scores.mean()
    std_score  = scores.std(ddof=1)
    n = len(scores)
    # 95 % CI with the t-distribution
    t_crit = stats.t.ppf(0.975, df=n - 1)
    margin = t_crit * (std_score / np.sqrt(n))
    ci_low, ci_high = mean_score - margin, mean_score + margin

    cv_results[name] = {
        'scores': scores, 'mean': mean_score, 'std': std_score,
        'ci_low': ci_low, 'ci_high': ci_high,
    }
    print(f'{name:22s}  F1 = {mean_score:.4f} ± {std_score:.4f}  '
          f'(95% CI: [{ci_low:.4f}, {ci_high:.4f}])  n={n}')

# ===========================================================================
# 3c. Statistical model comparison
# ===========================================================================
# We compare every pair of models using a paired t-test on the per-fold
# scores from the repeated CV.  Because both models see the *same* folds,
# the samples are paired, which gives more statistical power.

print('\n' + '=' * 70)
print('Paired t-test (per-fold F1 scores)')
print('=' * 70)

model_names = list(cv_results.keys())
for i in range(len(model_names)):
    for j in range(i + 1, len(model_names)):
        a, b = model_names[i], model_names[j]
        stat, p = stats.ttest_rel(cv_results[a]['scores'], cv_results[b]['scores'])
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        print(f'{a:22s}  vs  {b:22s}  '
              f't = {stat:+.3f}  p = {p:.4f}  {sig}')

# --- Wilcoxon signed-rank test (non-parametric alternative) -----------------
print('\n' + '=' * 70)
print('Wilcoxon signed-rank test (non-parametric)')
print('=' * 70)
for i in range(len(model_names)):
    for j in range(i + 1, len(model_names)):
        a, b = model_names[i], model_names[j]
        stat, p = stats.wilcoxon(cv_results[a]['scores'], cv_results[b]['scores'])
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        print(f'{a:22s}  vs  {b:22s}  '
              f'W = {stat:.1f}  p = {p:.4f}  {sig}')

print('\nSignificance: *** p<0.001  ** p<0.01  * p<0.05  ns = not significant')
print('\nModelling complete ✓')
